# Mistral-7B SAE — Kaggle Free (2×T4 or P100)

**Dense decoder SAE on Mistral-7B-v0.3, layer 16. ~5 hours on Kaggle free tier.**

---

This is the **Mistral-7B** variant of **Tier 2 — Explorer** of the `openinterp.org/train` ladder:

| Tier | Hardware | Model | Tokens | Wall time | Cost |
|------|----------|-------|--------|-----------|------|
| 1 — Hobbyist | Colab T4 | Gemma-2-2B | 20M | ~30 min | $0 |
| 2 — Explorer (Qwen3.5-4B) | Kaggle 2×T4 | Qwen3.5-4B | 150M | ~4–5 h | $0 |
| **2 — Explorer (this)** | **Kaggle 2×T4 or P100** | **Mistral-7B-v0.3** | **100M** | **~5 h** | **$0** |
| 3 — Paper-grade | Vast.ai B200 | Qwen3.5-9B / Gemma-4 | 1B+ | ~22 h | ~$30 |

Unlike the Qwen3.5 variant of this tier, **Mistral-7B is a standard dense decoder** (Llama-style: RMSNorm → attention → MLP → residual). There is no Gated Delta Network, no SSM state, no hybrid routing. Residual capture is trivial: `model.model.layers[16]` is a plain `nn.Module` and `output_hidden_states=True` returns a clean tensor per layer.

### Mistral-specific gotchas (small)

- **Sliding-window attention.** Mistral-v0.3 ships with a 4 096-token sliding-window attention mask (`config.sliding_window = 4096`). At our `SEQ_LEN = 1024` this never activates — every token sees the full prefix inside its sequence. Even at longer contexts, this does **not** affect SAE training: we capture the residual stream **after** attention writes back to it, so whatever attention computed (sliding or dense) is already baked into the signal we observe. Mentioned here only because people ask.
- **SentencePiece tokenizer.** Mistral uses a SentencePiece BPE tokenizer (32 768 vocab). `AutoTokenizer.from_pretrained` handles it transparently — no `trust_remote_code`, no chat template dance, no special imports. `sentencepiece` is preinstalled on Kaggle.
- **Gated HF repo.** `mistralai/Mistral-7B-v0.3` requires accepting the license on the HF model page and using an HF token that has read access. Your Kaggle `HF_TOKEN` secret must be from an account that has accepted it.

**Assumption**: you've completed Tier 1 (Gemma hobbyist) and know what TopK / AuxK / L0 / variance-explained mean.

**Kaggle quota**: Free tier gives 30 h/week of accelerator time. This notebook consumes ~5 h of that budget. Works on 2×T4 (preferred, puts SAE on GPU1) and on single P100 (slightly tighter on VRAM, still fits).

In [ ]:
# Cell 2 — Dependencies
# Mistral has been in transformers since v4.34, so no source install needed.
# We just pin the HF hub client and make sure datasets/safetensors are current.

import sys, subprocess

def pip(*a):
    return subprocess.run([sys.executable, '-m', 'pip', *a], check=False)

pip('install', '-q',
    'transformers>=4.45',
    'accelerate',
    'datasets',
    'huggingface_hub==1.5.0',
    'safetensors',
    'sentencepiece',
    'tokenizers',
    'protobuf',
    'einops')

import transformers, torch
print(f'transformers {transformers.__version__}')
print(f'torch        {torch.__version__}  —  CUDA {torch.version.cuda}  —  {torch.cuda.device_count()} GPU(s)')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  [{i}] {p.name}  {p.total_memory/1e9:.1f} GB')

## Config — 100M tokens in ~5 h on 2×T4

Ratios chosen for Mistral's `D_MODEL = 4096` on Kaggle's 16 GB T4s (or single 16 GB P100):

- `N_FEATURES = 65_536` = 16× expansion over `D_MODEL = 4096`
- `K_TOPK = 128` — ~0.2 % active features
- `K_AUX = 2048` with `ALPHA_AUX = 1/32` — dead-feature rescue
- `DEAD_TOKENS = 10M` — a feature is "dead" if unused for this many tokens
- `FWD_BATCH = 2 × SEQ_LEN 1024` on GPU0 for activations; SAE lives on GPU1

**Why 100M and not 150M?** Mistral-7B is 1.75× larger than Qwen3.5-4B, so each forward pass costs proportionally more VRAM and time. 100M tokens fits the same ~5 h Kaggle budget. Variance-explained typically saturates well before 100M anyway; this is comfortable.

**100M tokens / (FWD_BATCH 2 × SEQ_LEN 1024) ≈ 49 k forward passes ≈ ~5 h at ~2.7 passes/sec on 2×T4.**

In [ ]:
# Cell 4 — Config
MODEL_ID       = 'mistralai/Mistral-7B-v0.3'
LAYER          = 16                 # mid-stack. Mistral-7B has 32 layers.
D_MODEL        = 4096
N_FEATURES     = 65_536             # 16x expansion
K_TOPK         = 128
K_AUX          = 2048
ALPHA_AUX      = 1/32
DEAD_TOKENS    = 10_000_000
TOKEN_BUDGET   = 100_000_000
SEQ_LEN        = 1024
FWD_BATCH      = 2                  # sequences per model forward
BATCH_SIZE     = 4096               # SAE optimizer batch (token-level)
LR_PEAK        = 2e-4
LR_FLOOR       = 6e-5
WARMUP_STEPS   = 3000
CKPT_EVERY_TOK = 10_000_000         # HF checkpoint cadence (kernel-kill safe)

HF_USERNAME    = 'YOUR_HF_USERNAME'  # <-- EDIT
HF_REPO        = f'{HF_USERNAME}/mistral-7b-sae-L16'

print(f'Target: {TOKEN_BUDGET/1e6:.0f}M tokens into L{LAYER} SAE of {N_FEATURES} features, k={K_TOPK}')
print(f'Checkpointing to hf://{HF_REPO} every {CKPT_EVERY_TOK/1e6:.0f}M tokens')

## Auth — Kaggle Secrets

Add your HF token as a Kaggle Secret named `HF_TOKEN` (Add-ons → Secrets → Add Secret). A **write-scoped** token is required for checkpoint uploads, and the account that owns the token must have accepted the Mistral-7B-v0.3 license at <https://huggingface.co/mistralai/Mistral-7B-v0.3>.

In [ ]:
# Cell 6 — HF auth via Kaggle Secrets
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, HfApi, create_repo

hf_token = UserSecretsClient().get_secret('HF_TOKEN')
login(token=hf_token)
api = HfApi(token=hf_token)

try:
    create_repo(HF_REPO, repo_type='model', exist_ok=True, token=hf_token)
    print(f'Repo ready: https://huggingface.co/{HF_REPO}')
except Exception as e:
    print(f'repo warn: {e}')

## Load Mistral-7B-v0.3

Mistral is a standard `MistralForCausalLM` — no hybrid routing, no multimodal wrapper. `AutoModelForCausalLM` gives us a clean `model.model.layers[…]` path, and `output_hidden_states=True` returns per-layer residuals as expected. We use **SDPA** attention (not flash-attn; Kaggle T4 / P100 don't have flash-attn wheels preinstalled and we don't need them at `SEQ_LEN = 1024`).

`device_map='auto'` places the ~14 GB bf16 model on GPU0 (and spills to GPU1 only if needed on 2×T4). GPU1 is reserved for the SAE and its optimizer, which together take ~3 GB.

On **single-GPU P100**, the SAE co-locates with the model on GPU0 — still fits within 16 GB because the SAE is tiny relative to the model.

In [ ]:
# Cell 8 — Model load
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token)
if tok.pad_token_id is None:
    tok.pad_token_id = tok.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,             # NEVER torch_dtype= (transformers 5.x deprecation)
    device_map='auto',
    attn_implementation='sdpa',       # NEVER flash-attn on Kaggle T4/P100
    token=hf_token,
)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

torch.cuda.empty_cache()
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f'  GPU{i} after model load: {(total-free)/1e9:.1f} / {total/1e9:.1f} GB used')

print(f'model class:       {model.__class__.__name__}')
print(f'config model_type: {model.config.model_type}')
print(f'num layers:        {model.config.num_hidden_layers}')
print(f'hidden size:       {model.config.hidden_size}')
print(f'sliding_window:    {getattr(model.config, "sliding_window", None)}  (does not affect SAE training)')

# Sanity: verify the layer path exists and is the expected module type
assert hasattr(model.model.layers[LAYER], 'self_attn'), 'layer path mismatch'
print(f'layer {LAYER}:         {type(model.model.layers[LAYER]).__name__}')

## Residual capture via `output_hidden_states`

For a standard decoder, `hidden_states[0]` is the embedding output and decoder block `N` writes its residual to `hidden_states[N+1]`. So for `LAYER = 16` we grab `hidden_states[17]`.

We could equivalently attach a forward hook to `model.model.layers[16]`, but the `output_hidden_states=True` path is strictly equivalent here and keeps the code short. (On Qwen3.5-hybrid that API is the *only* robust option; on Mistral it's just the clean one.)

### Corpus mix
50 % **FineWeb-Edu** (broad web, high quality) + 50 % **OpenThoughts-114k** (reasoning traces). Reasoning bias is the point — we want features that track chain-of-thought structure, not just surface n-grams.

In [ ]:
# Cell 10 — Streaming dataset mix + hidden-state capture generator
import itertools, random
from datasets import load_dataset

LAYER_INDEX = LAYER + 1   # +1 because hidden_states[0] is embeddings

def open_streams(seed=0):
    try:
        fw = load_dataset(
            'HuggingFaceFW/fineweb-edu',
            name='sample-10BT',
            split='train',
            streaming=True,
        ).shuffle(seed=seed, buffer_size=2000)
    except Exception as e:
        print(f'fineweb-edu sample-10BT failed ({e}); falling back to CC-MAIN-2024-10')
        fw = load_dataset(
            'HuggingFaceFW/fineweb-edu',
            name='CC-MAIN-2024-10',
            split='train',
            streaming=True,
        ).shuffle(seed=seed, buffer_size=2000)

    ot = None
    for name in ['open-thoughts/OpenThoughts-114k']:
        try:
            ot = load_dataset(name, split='train', streaming=True).shuffle(seed=seed+1, buffer_size=2000)
            break
        except Exception as e:
            print(f'OT load {name} failed: {e}')
    if ot is None:
        print('WARN: no OpenThoughts, running 100% FineWeb-Edu')
    return fw, ot


def extract_text(sample):
    if 'text' in sample and isinstance(sample['text'], str):
        return sample['text']
    if 'conversations' in sample and isinstance(sample['conversations'], list):
        return '\n\n'.join(
            (m.get('value') or m.get('content') or '')
            for m in sample['conversations'] if isinstance(m, dict)
        )
    for k in ('content', 'output', 'response', 'answer'):
        if k in sample and isinstance(sample[k], str):
            return sample[k]
    return ''


def text_stream(seed=0):
    rng = random.Random(seed)
    fw, ot = open_streams(seed=seed)
    fw_it, ot_it = iter(fw), (iter(ot) if ot is not None else None)
    while True:
        src = fw_it if (ot_it is None or rng.random() < 0.5) else ot_it
        try:
            sample = next(src)
        except StopIteration:
            fw, ot = open_streams(seed=seed + rng.randint(1, 1_000_000))
            fw_it = iter(fw)
            ot_it = iter(ot) if ot is not None else None
            continue
        txt = extract_text(sample)
        if txt and len(txt) > 64:
            yield txt


def tokenize_pack(text_iter, seq_len=SEQ_LEN):
    '''Pack short docs end-to-end into fixed-length sequences.'''
    buf = []
    eos = tok.eos_token_id or 2  # Mistral eos is 2
    for txt in text_iter:
        ids = tok(txt, add_special_tokens=False).input_ids
        buf.extend(ids)
        buf.append(eos)
        while len(buf) >= seq_len:
            yield buf[:seq_len]
            buf = buf[seq_len:]


def batched(iterable, n):
    it = iter(iterable)
    while True:
        chunk = list(itertools.islice(it, n))
        if not chunk:
            return
        yield chunk


@torch.no_grad()
def activation_stream(seed=0):
    '''Yields (FWD_BATCH*SEQ_LEN, D_MODEL) fp32 residual tensors on SAE device.'''
    dev_model = next(model.parameters()).device
    dev_sae   = torch.device('cuda:1') if torch.cuda.device_count() > 1 else dev_model
    seqs = tokenize_pack(text_stream(seed=seed))
    for chunk in batched(seqs, FWD_BATCH):
        ids = torch.tensor(chunk, dtype=torch.long, device=dev_model)
        out = model(input_ids=ids, output_hidden_states=True, use_cache=False)
        h = out.hidden_states[LAYER_INDEX]            # (B, T, D)
        h = h.reshape(-1, h.shape[-1]).to(dev_sae, dtype=torch.float32)
        del out
        yield h


# Smoke test — one batch
it = activation_stream(seed=123)
sample_batch = next(it)
print(f'sample activation batch: {tuple(sample_batch.shape)}  dtype={sample_batch.dtype}  dev={sample_batch.device}')
print(f'mean={sample_batch.mean().item():+.4f}  std={sample_batch.std().item():.4f}  '
      f'abs_max={sample_batch.abs().max().item():.2f}')
assert sample_batch.shape[1] == D_MODEL, f'expected D_MODEL={D_MODEL}, got {sample_batch.shape[1]}'
del it, sample_batch
torch.cuda.empty_cache()

## TopK SAE + AuxK

Standard OpenAI-style TopK SAE:

- Encoder: `Linear(d → n) → bias → TopK(k)` — everything else is zeroed.
- Decoder: `Linear(n → d)` with unit-norm columns (re-normalized after each step).
- AuxK: among features that haven't fired in `DEAD_TOKENS`, take the top `K_AUX` pre-activations and use them to explain the residual error. Gradient flows back — this is how dead features come back to life.
- Geometric-median initialization for `b_dec` (Weiszfeld) to put the decoder bias near the data mean.

SAE weights stay in **fp32** even though the model is bf16 — the stability cost is cheap and T4 / P100 tensor cores work fine either way.

In [ ]:
# Cell 12 — TopK SAE with AuxK
import torch
import torch.nn as nn
import torch.nn.functional as F

class TopKSAE(nn.Module):
    def __init__(self, d=D_MODEL, n=N_FEATURES, k=K_TOPK, k_aux=K_AUX, dead_tokens=DEAD_TOKENS):
        super().__init__()
        self.d, self.n, self.k, self.k_aux = d, n, k, k_aux
        self.dead_tokens = dead_tokens

        self.W_enc = nn.Parameter(torch.empty(d, n))
        self.b_enc = nn.Parameter(torch.zeros(n))
        self.W_dec = nn.Parameter(torch.empty(n, d))
        self.b_dec = nn.Parameter(torch.zeros(d))

        nn.init.kaiming_uniform_(self.W_enc, a=5**0.5)
        with torch.no_grad():
            self.W_dec.copy_(self.W_enc.t().contiguous())
            self.renorm_decoder()

        self.register_buffer('last_fired', torch.zeros(n, dtype=torch.long))
        self.register_buffer('tokens_seen', torch.zeros(1, dtype=torch.long))

    @torch.no_grad()
    def renorm_decoder(self):
        nrm = self.W_dec.norm(dim=1, keepdim=True).clamp_min(1e-8)
        self.W_dec.div_(nrm)

    @torch.no_grad()
    def set_b_dec_geomedian(self, samples, iters=50, eps=1e-5):
        '''Weiszfeld iteration. samples: (N, d) fp32.'''
        x = samples.to(self.b_dec.device)
        mu = x.mean(0)
        for _ in range(iters):
            d = (x - mu).norm(dim=1).clamp_min(eps)
            w = 1.0 / d
            mu_new = (w[:, None] * x).sum(0) / w.sum()
            if (mu_new - mu).norm() < eps:
                break
            mu = mu_new
        self.b_dec.copy_(mu)

    def encode_pre(self, x):
        return (x - self.b_dec) @ self.W_enc + self.b_enc

    def forward(self, x):
        '''x: (B, d)  → (recon, aux_recon, z, topk_idx, pre)'''
        pre = self.encode_pre(x)
        topk_val, topk_idx = pre.topk(self.k, dim=-1)
        z = torch.zeros_like(pre)
        z.scatter_(-1, topk_idx, F.relu(topk_val))
        recon = z @ self.W_dec + self.b_dec

        aux_recon = None
        if self.training and self.k_aux > 0:
            dead_mask = (self.last_fired >= self.dead_tokens)
            n_dead = int(dead_mask.sum().item())
            if n_dead > 0:
                k_aux_eff = min(self.k_aux, n_dead)
                pre_dead = pre.masked_fill(~dead_mask, float('-inf'))
                aux_val, aux_idx = pre_dead.topk(k_aux_eff, dim=-1)
                z_aux = torch.zeros_like(pre)
                z_aux.scatter_(-1, aux_idx, F.relu(aux_val))
                aux_recon = z_aux @ self.W_dec

        return recon, aux_recon, z, topk_idx, pre

    @torch.no_grad()
    def update_fire_counter(self, topk_idx, batch_tokens):
        self.last_fired += batch_tokens
        fired = torch.unique(topk_idx)
        self.last_fired[fired] = 0
        self.tokens_seen += batch_tokens

    @torch.no_grad()
    def dead_count(self):
        return int((self.last_fired >= self.dead_tokens).sum().item())


print('TopKSAE defined.')

## Initialize + resume from HF checkpoint if one exists

Kaggle kernels can be killed (idle timeout, preemption, lost connection). To survive that:

1. We upload `sae_L16_resume.pt` (weights + optimizer + scheduler + step + tokens) every `CKPT_EVERY_TOK`.
2. On (re)start, we try to download it and pick up where we left off.
3. On fresh start, we initialize `b_dec` with a geometric median over ~64 k activations.

In [ ]:
# Cell 14 — Instantiate SAE, optimizer; try resume
import math, os, json
from pathlib import Path
from huggingface_hub import hf_hub_download

dev_sae = torch.device('cuda:1') if torch.cuda.device_count() > 1 else torch.device('cuda:0')
print(f'SAE device: {dev_sae}')

sae = TopKSAE().to(dev_sae, dtype=torch.float32)
optim = torch.optim.Adam(sae.parameters(), lr=LR_PEAK, betas=(0.9, 0.999), eps=1e-8)

TOTAL_STEPS = max(1, TOKEN_BUDGET // BATCH_SIZE)

def lr_at(step):
    if step < WARMUP_STEPS:
        return LR_PEAK * step / max(1, WARMUP_STEPS)
    prog = (step - WARMUP_STEPS) / max(1, TOTAL_STEPS - WARMUP_STEPS)
    prog = min(1.0, max(0.0, prog))
    return LR_FLOOR + 0.5 * (LR_PEAK - LR_FLOOR) * (1 + math.cos(math.pi * prog))

state = {'step': 0, 'tokens_seen': 0}
resumed = False
try:
    local = hf_hub_download(repo_id=HF_REPO, filename='sae_L16_resume.pt', token=hf_token)
    ckpt = torch.load(local, map_location='cpu', weights_only=False)
    sae.load_state_dict(ckpt['sae'])
    optim.load_state_dict(ckpt['optim'])
    state['step'] = int(ckpt.get('step', 0))
    state['tokens_seen'] = int(ckpt.get('tokens_seen', 0))
    sae.to(dev_sae, dtype=torch.float32)
    print(f'Resumed from step {state["step"]}, tokens_seen {state["tokens_seen"]/1e6:.1f}M')
    resumed = True
except Exception as e:
    print(f'No resume file on HF ({e}); fresh init with geometric median.')

if not resumed:
    geo_batches = []
    n_needed = 65536
    n_have = 0
    it = activation_stream(seed=42)
    while n_have < n_needed:
        b = next(it)
        geo_batches.append(b)
        n_have += b.shape[0]
    gm_samples = torch.cat(geo_batches, dim=0)[:n_needed]
    sae.set_b_dec_geomedian(gm_samples, iters=50)
    print(f'b_dec initialised via geom-median on {n_needed} tokens — '
          f'norm={sae.b_dec.norm().item():.3f}')
    del geo_batches, gm_samples, it
    torch.cuda.empty_cache()

## Training loop

We pull activation batches of `FWD_BATCH * SEQ_LEN = 2048` tokens from the model, buffer them, and feed `BATCH_SIZE = 4096`-token SAE optimizer steps (so one optimizer step = two model forwards).

Loss:

$$ \mathcal{L} = \underbrace{\lVert x - \hat{x} \rVert_2^2}_{\text{recon}} + \alpha_\text{aux} \underbrace{\lVert (x - \hat{x}) - \hat{x}_\text{aux} \rVert_2^2}_{\text{auxk on residual}} $$

Checkpoints go to HF every `CKPT_EVERY_TOK`. We log variance-explained, L0 (should equal K_TOPK), and dead-feature count. The trailing decoder-gradient projection keeps columns on the unit sphere (tangent-space update, Bricken et al.).

In [ ]:
# Cell 16 — Training loop + final checkpoint + 500k held-out validation
import io, json, time
from pathlib import Path
from safetensors.torch import save_file as save_safetensors
from huggingface_hub import upload_file, delete_file
from tqdm.auto import tqdm

TMP = Path('/kaggle/working/ckpt'); TMP.mkdir(exist_ok=True)

def save_weights_safetensors(sae, path):
    sd = {k: v.detach().cpu().contiguous() for k, v in sae.state_dict().items()
          if not k.startswith(('last_fired', 'tokens_seen'))}
    save_safetensors(sd, str(path))

def save_resume(path, sae, optim, step, tokens_seen):
    torch.save({
        'sae': sae.state_dict(),
        'optim': optim.state_dict(),
        'step': step,
        'tokens_seen': tokens_seen,
    }, path)

def push_checkpoint(sae, optim, step, tokens_seen, is_final=False):
    w_path   = TMP / 'sae_L16_latest.safetensors'
    r_path   = TMP / 'sae_L16_resume.pt'
    cfg_path = TMP / 'cfg.json'
    save_weights_safetensors(sae, w_path)
    cfg = dict(
        model_id=MODEL_ID, layer=LAYER, d_model=D_MODEL, n_features=N_FEATURES,
        k_topk=K_TOPK, k_aux=K_AUX, alpha_aux=ALPHA_AUX,
        dead_tokens=DEAD_TOKENS, seq_len=SEQ_LEN,
        batch_size=BATCH_SIZE, token_budget=TOKEN_BUDGET,
        step=step, tokens_seen=tokens_seen, final=is_final,
    )
    cfg_path.write_text(json.dumps(cfg, indent=2))
    upload_file(path_or_fileobj=str(w_path), path_in_repo='sae_L16_latest.safetensors',
                repo_id=HF_REPO, token=hf_token)
    upload_file(path_or_fileobj=str(cfg_path), path_in_repo='cfg.json',
                repo_id=HF_REPO, token=hf_token)
    if not is_final:
        save_resume(r_path, sae, optim, step, tokens_seen)
        upload_file(path_or_fileobj=str(r_path), path_in_repo='sae_L16_resume.pt',
                    repo_id=HF_REPO, token=hf_token)

step = state['step']
tokens_seen = state['tokens_seen']
next_ckpt_at = ((tokens_seen // CKPT_EVERY_TOK) + 1) * CKPT_EVERY_TOK

pbar = tqdm(total=TOKEN_BUDGET, initial=tokens_seen, desc='training', unit='tok',
            unit_scale=True, smoothing=0.05)

acts_iter = activation_stream(seed=1000 + step)
acts_buf = torch.empty(0, D_MODEL, device=dev_sae, dtype=torch.float32)
running = {'recon': 0.0, 'aux': 0.0, 've': 0.0, 'n': 0}
t0 = time.time()

try:
    sae.train()
    while tokens_seen < TOKEN_BUDGET:
        while acts_buf.shape[0] < BATCH_SIZE:
            chunk = next(acts_iter)
            acts_buf = torch.cat([acts_buf, chunk], dim=0)
        x = acts_buf[:BATCH_SIZE]
        acts_buf = acts_buf[BATCH_SIZE:]

        lr = lr_at(step)
        for g in optim.param_groups:
            g['lr'] = lr

        recon, aux_recon, z, topk_idx, pre = sae(x)
        err = x - recon
        recon_loss = err.pow(2).mean()
        if aux_recon is not None:
            aux_err = err.detach() - aux_recon
            aux_loss = aux_err.pow(2).mean()
            loss = recon_loss + ALPHA_AUX * aux_loss
        else:
            aux_loss = torch.tensor(0.0, device=dev_sae)
            loss = recon_loss

        optim.zero_grad(set_to_none=True)
        loss.backward()
        # Zero the gradient along each decoder column's own direction (keep unit-norm)
        with torch.no_grad():
            if sae.W_dec.grad is not None:
                proj = (sae.W_dec.grad * sae.W_dec).sum(dim=1, keepdim=True)
                sae.W_dec.grad.sub_(proj * sae.W_dec)
        optim.step()
        with torch.no_grad():
            sae.renorm_decoder()
            sae.update_fire_counter(topk_idx, batch_tokens=BATCH_SIZE)

        with torch.no_grad():
            var_x = x.var(unbiased=False).clamp_min(1e-8)
            ve = 1.0 - err.pow(2).mean() / var_x
            running['recon'] += recon_loss.item()
            running['aux']   += aux_loss.item() if aux_recon is not None else 0.0
            running['ve']    += ve.item()
            running['n']     += 1

        step += 1
        tokens_seen += BATCH_SIZE
        pbar.update(BATCH_SIZE)

        if step % 25 == 0:
            n = running['n']
            pbar.set_postfix(
                lr=f'{lr:.2e}',
                recon=f'{running["recon"]/n:.4f}',
                aux=f'{running["aux"]/n:.4f}',
                ve=f'{running["ve"]/n:.3f}',
                dead=sae.dead_count(),
                L0=K_TOPK,
            )
            running = {'recon': 0.0, 'aux': 0.0, 've': 0.0, 'n': 0}

        if tokens_seen >= next_ckpt_at:
            print(f'\n[ckpt @ {tokens_seen/1e6:.1f}M tokens, step {step}] uploading…')
            try:
                push_checkpoint(sae, optim, step, tokens_seen, is_final=False)
                print('[ckpt] done.')
            except Exception as e:
                print(f'[ckpt] upload failed, will retry next window: {e}')
            next_ckpt_at += CKPT_EVERY_TOK

finally:
    pbar.close()
    elapsed = time.time() - t0
    print(f'elapsed {elapsed/3600:.2f} h  —  tokens_seen {tokens_seen/1e6:.1f}M  —  step {step}')

# ─── Final upload ──────────────────────────────────────────────────────────────
print('\nUploading final weights…')
try:
    push_checkpoint(sae, optim, step, tokens_seen, is_final=True)
    print('Final weights uploaded.')
except Exception as e:
    print(f'final upload error: {e}')

try:
    delete_file(path_in_repo='sae_L16_resume.pt', repo_id=HF_REPO, token=hf_token)
    print('resume.pt deleted.')
except Exception as e:
    print(f'resume delete warn: {e}')

# ─── Held-out validation on 500k fresh tokens ──────────────────────────────────
print('\nValidation over 500k held-out tokens…')
sae.eval()
val_tokens = 500_000
val_seen = 0
val_recon_sse = 0.0
val_var_sum  = 0.0
L0_sum = 0.0
fired_ever = torch.zeros(N_FEATURES, dtype=torch.bool, device=dev_sae)

val_iter = activation_stream(seed=99999)
with torch.no_grad():
    while val_seen < val_tokens:
        chunk = next(val_iter)
        x = chunk[:min(chunk.shape[0], val_tokens - val_seen)]
        recon, _, z, topk_idx, _ = sae(x)
        err = x - recon
        val_recon_sse += err.pow(2).sum().item()
        val_var_sum   += (x - x.mean(0, keepdim=True)).pow(2).sum().item()
        L0_sum        += (z > 0).float().sum(dim=-1).sum().item()
        fired_ever[torch.unique(topk_idx)] = True
        val_seen += x.shape[0]

ve_val   = 1.0 - val_recon_sse / max(1e-9, val_var_sum)
L0_val   = L0_sum / max(1, val_seen)
dead_val = int((~fired_ever).sum().item())

report = dict(
    model_id=MODEL_ID, layer=LAYER,
    tokens_trained=tokens_seen, steps=step,
    val_tokens=val_seen,
    val_variance_explained=ve_val,
    val_L0=L0_val,
    val_dead_features=dead_val,
    val_dead_frac=dead_val / N_FEATURES,
    k_topk=K_TOPK, n_features=N_FEATURES,
)
print(json.dumps(report, indent=2))

rpath = TMP / 'val_report.json'
rpath.write_text(json.dumps(report, indent=2))
try:
    upload_file(path_or_fileobj=str(rpath), path_in_repo='val_report.json',
                repo_id=HF_REPO, token=hf_token)
    print(f'val_report.json uploaded to https://huggingface.co/{HF_REPO}')
except Exception as e:
    print(f'val report upload warn: {e}')

print('\nDone. Next tier: Vast.ai B200 for 1B tokens on Qwen3.5-9B or Gemma-4.')